# Dataset 1 Thermal Training and Evaluation

This notebook creates a **Dataset 1-only** derived YOLO dataset, trains a one-class `drone` detector, selects an image-level decision threshold using validation data, and evaluates the selected checkpoint once on a recording-disjoint held-out test split.

**Use this experiment for:** Dataset 1 internal generalization. It is not a drone-versus-bird or cross-dataset evaluation.

Dataset 1's supplied `track1_test` and `track2_test` folders are inference-only: they have no compatible frame-level labels and are deliberately excluded. The split below is constructed only from the 223 labelled sequences in `train`.

Valid `exist == 1` frames with valid `gt_rect` boxes are drone-positive examples. `exist != 1` frames are retained as empty-label target-absent backgrounds. Invalid positive annotations and unreadable source images are excluded rather than converted to negatives. Results therefore measure Dataset 1 internal generalization, not drone-versus-bird discrimination or cross-dataset generalization.

In [1]:
from __future__ import annotations

from collections import Counter
from datetime import datetime
from pathlib import Path
import json
import random
import shutil
import sys

import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image, UnidentifiedImageError
from ultralytics import YOLO

WORKING_DIRECTORY = Path.cwd().resolve()
if (WORKING_DIRECTORY / '.git').is_dir():
    REPO_ROOT = WORKING_DIRECTORY
elif (WORKING_DIRECTORY.parent / '.git').is_dir():
    REPO_ROOT = WORKING_DIRECTORY.parent
else:
    raise RuntimeError('Open this notebook from the repository root or notebooks/ directory.')

THERMAL_ROOT = REPO_ROOT / 'data' / 'thermal-ir'
DATASET1_TRAIN_ROOT = THERMAL_ROOT / 'raw' / 'anti_uav_dataset1' / 'train'
DERIVED_ROOT = THERMAL_ROOT / 'processed' / 'dataset1_only_yolo_v1'
RESULTS_ROOT = REPO_ROOT / 'results' / 'thermal-ir'
EXPERIMENT_NAME = 'experiment_c_dataset1_only'
RANDOM_SEED = 590
FRAME_STRIDE = 5
SPLIT_RATIOS = {'train': 0.70, 'val': 0.15, 'test': 0.15}
CLASS_NAMES = {0: 'drone'}

print(f'Python: {sys.version.split()[0]}')
print(f'Repository: {REPO_ROOT}')
print(f'Dataset 1 train root: {DATASET1_TRAIN_ROOT}')


Python: 3.11.7
Repository: D:\Capstone\aai-590-capstone
Dataset 1 train root: D:\Capstone\aai-590-capstone\data\thermal-ir\raw\anti_uav_dataset1\train


## Source-data audit and eligibility

This audit reads source files only. It requires the Dataset 1 `train` partition to contain 223 sequences and fails if an annotation list is not aligned with its source frames. Bounding boxes with non-positive dimensions or coordinates beyond the image are logged and excluded from the derived export.

In [2]:
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png'}

def source_frames(sequence_dir: Path) -> list[Path]:
    return sorted((p for p in sequence_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES), key=lambda p: p.name)

def xywh_issue(box, width: int, height: int) -> str | None:
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        return 'invalid_box_shape'
    try:
        x, y, box_width, box_height = (float(value) for value in box)
    except (TypeError, ValueError):
        return 'non_numeric_box'
    if not all(np.isfinite(value) for value in (x, y, box_width, box_height)):
        return 'non_finite_box'
    if box_width <= 0 or box_height <= 0:
        return 'non_positive_box_dimensions'
    if x < 0 or y < 0:
        return 'negative_box_origin'
    if x + box_width > width or y + box_height > height:
        return 'box_exceeds_image_boundary'
    return None

def inspect_dataset1_train() -> tuple[pd.DataFrame, pd.DataFrame]:
    if not DATASET1_TRAIN_ROOT.is_dir():
        raise FileNotFoundError(f'Dataset 1 train directory is missing: {DATASET1_TRAIN_ROOT}')
    sequence_dirs = sorted(p for p in DATASET1_TRAIN_ROOT.iterdir() if p.is_dir())
    if len(sequence_dirs) != 223:
        raise RuntimeError(f'Expected 223 Dataset 1 train sequences, found {len(sequence_dirs)}.')
    sequences, issues = [], []
    for sequence_dir in sequence_dirs:
        frames = source_frames(sequence_dir)
        label_path = sequence_dir / 'IR_label.json'
        if not frames or not label_path.is_file():
            raise RuntimeError(f'{sequence_dir.name} is missing source frames or IR_label.json.')
        labels = json.loads(label_path.read_text(encoding='utf-8'))
        exists, boxes = labels.get('exist'), labels.get('gt_rect')
        if not isinstance(exists, list) or not isinstance(boxes, list):
            raise RuntimeError(f'{sequence_dir.name} does not provide list-valued exist and gt_rect annotations.')
        if not (len(frames) == len(exists) == len(boxes)):
            raise RuntimeError(f'{sequence_dir.name} has a frame/annotation length mismatch: {len(frames)}, {len(exists)}, {len(boxes)}.')
        with Image.open(frames[0]) as first_image:
            width, height = first_image.size
        valid_positive = 0
        target_absent = 0
        for index, (exists_flag, box, frame_path) in enumerate(zip(exists, boxes, frames), start=1):
            if frame_path.stat().st_size == 0:
                issues.append({'source_sequence': sequence_dir.name, 'original_frame': index, 'issue_type': 'unreadable_source_frame', 'detail': 'zero-byte image'})
                continue
            if exists_flag == 1:
                issue = xywh_issue(box, width, height)
                if issue:
                    issues.append({'source_sequence': sequence_dir.name, 'original_frame': index, 'issue_type': issue, 'detail': json.dumps(box)})
                else:
                    valid_positive += 1
            else:
                target_absent += 1
        sequences.append({'source_dataset': 'anti_uav_dataset1', 'source_sequence': sequence_dir.name, 'frame_count': len(frames), 'image_width': width, 'image_height': height, 'valid_drone_frames': valid_positive, 'target_absent_frames': target_absent, 'annotation_path': label_path.relative_to(REPO_ROOT).as_posix()})
    return pd.DataFrame(sequences), pd.DataFrame(issues)

sequence_manifest, issue_log = inspect_dataset1_train()
display(sequence_manifest.describe())
display(issue_log['issue_type'].value_counts().rename_axis('issue_type').to_frame('count'))


,frame_count,image_width,image_height,valid_drone_frames,target_absent_frames
count,223.000000,223.000000,223.0,223.000000,223.000000
mean,1110.219731,626.798206,512.0,1029.663677,78.905830
std,335.239367,39.017547,0.0,335.259264,197.595376
min,28.000000,512.000000,512.0,0.000000,0.000000
25%,949.000000,640.000000,512.0,913.000000,0.000000
50%,1000.000000,640.000000,512.0,1000.000000,0.000000
75%,1500.000000,640.000000,512.0,1353.000000,60.500000
max,1630.000000,640.000000,512.0,1500.000000,1500.000000


,count
issue_type,
box_exceeds_image_boundary,130
non_positive_box_dimensions,121
unreadable_source_frame,117


## Recording-disjoint split and YOLO export

Each whole Dataset 1 sequence is assigned once to train, validation, or test with a fixed seed. Frame-level random splitting is prohibited because neighboring video frames are strongly dependent. There is no separate recording-level bird class in Dataset 1 to stratify; both valid drone and target-absent frames from a sequence stay together in its assigned partition.

In [3]:
def assign_sequence_splits(manifest: pd.DataFrame) -> dict[str, str]:
    members = manifest.source_sequence.tolist()
    random.Random(f'{RANDOM_SEED}:dataset1').shuffle(members)
    train_end = round(len(members) * SPLIT_RATIOS['train'])
    val_end = train_end + round(len(members) * SPLIT_RATIOS['val'])
    return {sequence: ('train' if index < train_end else 'val' if index < val_end else 'test') for index, sequence in enumerate(members)}

def normalised_drone_label(box, width: int, height: int) -> str:
    x, y, box_width, box_height = (float(value) for value in box)
    return f'0 {(x + box_width / 2) / width:.8f} {(y + box_height / 2) / height:.8f} {box_width / width:.8f} {box_height / height:.8f}\n'

def build_dataset1_yolo_export(archive_existing: bool = False) -> pd.DataFrame:
    if DERIVED_ROOT.exists() and any(DERIVED_ROOT.iterdir()):
        if not archive_existing:
            raise RuntimeError(f'{DERIVED_ROOT} is not empty. Set archive_existing=True to preserve it and rebuild.')
        archive_path = DERIVED_ROOT.with_name(f'{DERIVED_ROOT.name}_archived_{datetime.now():%Y%m%d_%H%M%S}')
        DERIVED_ROOT.rename(archive_path)
        print(f'Archived prior derived export to {archive_path}')
    assignments = assign_sequence_splits(sequence_manifest)
    invalid_positive = {(row.source_sequence, int(row.original_frame)) for _, row in issue_log.iterrows()}
    for split in SPLIT_RATIOS:
        (DERIVED_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
        (DERIVED_ROOT / 'labels' / split).mkdir(parents=True, exist_ok=True)
    records, skipped = [], []
    for _, sequence_row in sequence_manifest.iterrows():
        sequence = sequence_row.source_sequence
        split = assignments[sequence]
        labels = json.loads((REPO_ROOT / sequence_row.annotation_path).read_text(encoding='utf-8'))
        for frame_number, (exists_flag, box, source_path) in enumerate(zip(labels['exist'], labels['gt_rect'], source_frames(DATASET1_TRAIN_ROOT / sequence)), start=1):
            if frame_number % FRAME_STRIDE or (sequence, frame_number) in invalid_positive:
                continue
            is_positive = exists_flag == 1
            if is_positive and xywh_issue(box, int(sequence_row.image_width), int(sequence_row.image_height)) is not None:
                continue
            stem = f'anti_uav_dataset1_{sequence}_{frame_number:06d}'
            image_path = DERIVED_ROOT / 'images' / split / f'{stem}.jpg'
            label_path = DERIVED_ROOT / 'labels' / split / f'{stem}.txt'
            try:
                with Image.open(source_path) as image:
                    image.load()
                    image.convert('RGB').save(image_path, quality=95)
                    label_text = normalised_drone_label(box, image.width, image.height) if is_positive else ''
            except (UnidentifiedImageError, OSError) as error:
                skipped.append({'source_sequence': sequence, 'original_frame': frame_number, 'source_path': source_path.relative_to(REPO_ROOT).as_posix(), 'reason': str(error)})
                continue
            label_path.write_text(label_text, encoding='utf-8')
            records.append({'source_dataset': 'anti_uav_dataset1', 'source_sequence': sequence, 'original_frame': frame_number, 'source_class': 'DRONE' if is_positive else 'TARGET_ABSENT', 'export_policy': 'positive_drone' if is_positive else 'negative_target_absent', 'label_status': 'drone_positive' if is_positive else 'no_drone', 'cleaning_action': 'exported_valid_drone' if is_positive else 'exported_target_absent_background', 'split': split, 'image_path': image_path.relative_to(REPO_ROOT).as_posix(), 'label_path': label_path.relative_to(REPO_ROOT).as_posix()})
    frames = pd.DataFrame(records)
    if frames.empty or not (frames.label_status == 'drone_positive').any() or not (frames.label_status == 'no_drone').any():
        raise RuntimeError('Export must contain both valid drone-positive and target-absent frames.')
    leakage = frames.groupby('source_sequence').split.nunique()
    if (leakage != 1).any():
        raise RuntimeError('Split leakage detected: at least one sequence appears in multiple partitions.')
    metadata_root = DERIVED_ROOT / 'metadata'
    metadata_root.mkdir(exist_ok=True)
    frames.to_csv(metadata_root / 'frames.csv', index=False)
    issue_log.to_csv(metadata_root / 'annotation_issues.csv', index=False)
    pd.DataFrame(skipped).to_csv(metadata_root / 'skipped_unreadable_source_frames.csv', index=False)
    sequence_manifest.assign(split=sequence_manifest.source_sequence.map(assignments)).to_csv(metadata_root / 'sequence_manifest.csv', index=False)
    report = {'frame_stride': FRAME_STRIDE, 'split_ratios': SPLIT_RATIOS, 'random_seed': RANDOM_SEED, 'recording_disjoint': True, 'frame_counts': frames.groupby(['split', 'export_policy']).size().reset_index(name='frames').to_dict('records'), 'sequence_counts': frames.drop_duplicates('source_sequence').groupby('split').size().to_dict(), 'skipped_unreadable_export_frames': len(skipped)}
    (metadata_root / 'split_integrity_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    data_yaml = {'path': str(DERIVED_ROOT), 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'names': CLASS_NAMES}
    with (DERIVED_ROOT / f'{EXPERIMENT_NAME}.yaml').open('w', encoding='utf-8') as stream:
        yaml.safe_dump(data_yaml, stream, sort_keys=False)
    return frames

frames_path = DERIVED_ROOT / 'metadata' / 'frames.csv'
integrity_path = DERIVED_ROOT / 'metadata' / 'split_integrity_report.json'
frames = pd.read_csv(frames_path) if frames_path.is_file() and integrity_path.is_file() else build_dataset1_yolo_export(archive_existing=DERIVED_ROOT.exists())
display(frames.groupby(['split', 'export_policy', 'label_status']).size().rename('frames').to_frame())
display(frames.drop_duplicates('source_sequence').groupby('split').size().rename('sequences').to_frame())


requirements: Ultralytics requirement ['pi-heif'] not found, attempting AutoUpdate...
Using Python 3.11.7 environment at: d:\Capstone\.venv
Resolved 2 packages in 157ms
Prepared 1 package in 144ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 145ms
 + pi-heif==1.4.0

requirements: AutoUpdate success  0.7s
WARNING requirements: Restart runtime or rerun command for updates to take effect



frames
split export_policy          label_status          
test  negative_target_absent no_drone           610
      positive_drone         drone_positive    6632
train negative_target_absent no_drone          2499
      positive_drone         drone_positive   31822
val   negative_target_absent no_drone           414
      positive_drone         drone_positive    7436

,sequences
split,
test,34
train,156
val,33


## Training

The configuration intentionally mirrors the Dataset 2 baseline where applicable: pretrained YOLO nano, 640-pixel inputs, AdamW, fixed seed, and thermal-appropriate geometric augmentation without HSV color changes. The held-out test partition is not used during training, early stopping, or threshold selection.

In [4]:
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Install the CUDA-enabled PyTorch build, then restart the notebook kernel.')
print(f'PyTorch: {torch.__version__}; CUDA: {torch.version.cuda}; device: {torch.cuda.get_device_name(0)}')

DATA_YAML = DERIVED_ROOT / f'{EXPERIMENT_NAME}.yaml'
RUN_ROOT = RESULTS_ROOT / 'runs'
RUN_NAME = f'{EXPERIMENT_NAME}_batch32'
MODEL_CONFIG = {
    'model': 'yolo26n.pt', 'imgsz': 640, 'epochs': 100, 'batch': 32, 'device': 0,
    'optimizer': 'AdamW', 'lr0': 1e-3, 'weight_decay': 5e-4, 'patience': 20,
    'seed': RANDOM_SEED, 'deterministic': True, 'workers': 8, 'pretrained': True, 'cache': False,
    'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0, 'fliplr': 0.5, 'flipud': 0.0,
    'degrees': 0.0, 'translate': 0.10, 'scale': 0.25, 'mosaic': 0.5, 'close_mosaic': 10,
}

def train_dataset1() -> Path:
    required = [DATA_YAML, DERIVED_ROOT / 'metadata' / 'frames.csv', DERIVED_ROOT / 'metadata' / 'split_integrity_report.json']
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise RuntimeError('Training is blocked until the validated Dataset 1 export exists: ' + ', '.join(missing))
    model = YOLO(MODEL_CONFIG['model'])
    model.train(data=str(DATA_YAML), project=str(RUN_ROOT), name=RUN_NAME, exist_ok=False, **{key: value for key, value in MODEL_CONFIG.items() if key != 'model'})
    return Path(model.trainer.save_dir)


PyTorch: 2.14.1+cu130; CUDA: 13.0; device: NVIDIA GeForce RTX 5080


In [5]:
# Run this cell once to train. Do not reuse an existing run directory for a new experiment.
training_run_dir = train_dataset1()
BEST_WEIGHTS = training_run_dir / 'weights' / 'best.pt'
if not BEST_WEIGHTS.is_file():
    raise RuntimeError(f'Expected best checkpoint was not written: {BEST_WEIGHTS}')
print(f'Completed training run: {training_run_dir}')


New https://pypi.org/project/ultralytics/8.4.213 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.198  Python-3.11.7 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5080, 16276MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Capstone\aai-590-capstone\data\thermal-ir\processed\dataset1_only_yolo_v1\experiment_c_dataset1_only.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0

## Validation-selected threshold and held-out test evaluation

Detection mAP is calculated by Ultralytics on the held-out test partition. The notebook separately converts predictions into an image-level drone-present/no-drone decision. It selects the confidence threshold using only validation images, minimizing false positives while retaining at least 90% validation recall; then it locks that threshold for one held-out test evaluation. Dataset 1 target-absent backgrounds are useful negatives, but they are not bird hard negatives.

In [6]:
TEST_CONFIDENCE_THRESHOLD = 0.001
DECISION_THRESHOLDS = (0.05, 0.10, 0.25, 0.50, 0.75)
MIN_VALIDATION_RECALL = 0.90

def predictions_for_split(model: YOLO, split: str) -> pd.DataFrame:
    split_rows = frames.loc[frames.split.eq(split)].copy()
    observations = []
    for _, row in split_rows.iterrows():
        prediction = model.predict(str(REPO_ROOT / row.image_path), conf=TEST_CONFIDENCE_THRESHOLD, verbose=False)[0]
        confidences = [float(value) for value in prediction.boxes.conf]
        observations.append({**row.to_dict(), 'ground_truth_drone_present': row.label_status == 'drone_positive', 'max_confidence': max(confidences, default=0.0), 'prediction_count': len(confidences)})
    return pd.DataFrame(observations)

def decision_metrics(scores: pd.DataFrame, threshold: float) -> tuple[pd.DataFrame, dict]:
    evaluated = scores.copy()
    evaluated['predicted_drone_present'] = evaluated.max_confidence.ge(threshold)
    evaluated['binary_outcome'] = np.select([evaluated.ground_truth_drone_present & evaluated.predicted_drone_present, ~evaluated.ground_truth_drone_present & evaluated.predicted_drone_present, evaluated.ground_truth_drone_present & ~evaluated.predicted_drone_present], ['true_positive', 'false_positive', 'false_negative'], default='true_negative')
    tp, fp, fn, tn = (int((evaluated.binary_outcome == name).sum()) for name in ('true_positive', 'false_positive', 'false_negative', 'true_negative'))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return evaluated, {'decision_threshold': threshold, 'true_positive': tp, 'false_positive': fp, 'false_negative': fn, 'true_negative': tn, 'precision': precision, 'recall': recall, 'f1': 2 * precision * recall / (precision + recall) if precision + recall else 0.0, 'target_absent_false_positive_rate': fp / (fp + tn) if fp + tn else 0.0}

def select_threshold(validation_scores: pd.DataFrame) -> tuple[float, pd.DataFrame]:
    summaries = []
    for threshold in DECISION_THRESHOLDS:
        _, metrics = decision_metrics(validation_scores, threshold)
        summaries.append(metrics)
    report = pd.DataFrame(summaries).sort_values('decision_threshold')
    eligible = report.loc[report.recall.ge(MIN_VALIDATION_RECALL)]
    candidates = eligible if not eligible.empty else report
    selected = candidates.sort_values(['target_absent_false_positive_rate', 'f1', 'decision_threshold'], ascending=[True, False, False]).iloc[0]
    return float(selected.decision_threshold), report

# If resuming a completed run in a fresh kernel, set this path explicitly before evaluating:
# BEST_WEIGHTS = RUN_ROOT / RUN_NAME / 'weights' / 'best.pt'
model = YOLO(str(BEST_WEIGHTS))
validation_scores = predictions_for_split(model, 'val')
DECISION_THRESHOLD, validation_report = select_threshold(validation_scores)
display(validation_report)
print(f'Validation-selected threshold: {DECISION_THRESHOLD:.2f}')


,decision_threshold,true_positive,false_positive,false_negative,true_negative,precision,recall,f1,target_absent_false_positive_rate
0,0.05,6320,47,1116,367,0.992618,0.849919,0.915743,0.113527
1,0.10,6097,34,1339,380,0.994454,0.819930,0.898799,0.082126
2,0.25,5771,16,1665,398,0.997235,0.776089,0.872873,0.038647
3,0.50,5043,1,2393,413,0.999802,0.678187,0.808173,0.002415
4,0.75,2582,0,4854,414,1.000000,0.347230,0.515472,0.000000


Validation-selected threshold: 0.75


In [7]:
evaluation_root = RESULTS_ROOT / 'evaluation' / RUN_NAME
evaluation_root.mkdir(parents=True, exist_ok=True)

# Ultralytics supplies held-out object-detection mAP and curves.
detection_results = model.val(data=str(DATA_YAML), split='test', conf=TEST_CONFIDENCE_THRESHOLD, plots=True)
(evaluation_root / 'test_detection_metrics.json').write_text(json.dumps(detection_results.results_dict, indent=2), encoding='utf-8')

test_scores = predictions_for_split(model, 'test')
test_decisions, test_summary = decision_metrics(test_scores, DECISION_THRESHOLD)
validation_scores.to_csv(evaluation_root / 'validation_image_scores.csv', index=False)
validation_report.to_csv(evaluation_root / 'validation_threshold_report.csv', index=False)
test_decisions.to_csv(evaluation_root / 'test_image_decisions.csv', index=False)
(evaluation_root / 'test_decision_metrics.json').write_text(json.dumps(test_summary, indent=2), encoding='utf-8')

display(pd.Series(test_summary))
print(f'Wrote evaluation artifacts to: {evaluation_root}')
print('Interpret the target-absent false-positive rate as Dataset 1 background rejection only; it is not a bird false-positive rate.')


Ultralytics 8.4.198  Python-3.11.7 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5080, 16276MiB)
YOLO26n summary (fused): 121 layers, 2,389,735 parameters, 0 gradients, 5.4 GFLOPs
WARNING val: Slow image access detected (ping: 0.2±0.0 ms, read: 11.0±6.7 MB/s, size: 43.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning D:\Capstone\aai-590-capstone\data\thermal-ir\processed\dataset1_only_yolo_v1\labels\test... 7242 images, 610 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 7242/7242 1.2Kit/s 6.0s0.0s
val: New cache created: D:\Capstone\aai-590-capstone\data\thermal-ir\processed\dataset1_only_yolo_v1\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 453/453 12.7it/s 35.8s0.1ss
                   all       7242       6632      0.898      0.717      0.773      0.402
Speed: 0.5ms preprocess, 0.7ms inference, 0.0ms los

decision_threshold                      0.750000
true_positive                        2350.000000
false_positive                          0.000000
false_negative                       4282.000000
true_negative                         610.000000
precision                               1.000000
recall                                  0.354343
f1                                      0.523269
target_absent_false_positive_rate       0.000000
dtype: float64

Wrote evaluation artifacts to: D:\Capstone\aai-590-capstone\results\thermal-ir\evaluation\experiment_c_dataset1_only_batch32
Interpret the target-absent false-positive rate as Dataset 1 background rejection only; it is not a bird false-positive rate.


## Interpretation checklist

Report: (1) the derived frame and sequence counts by split, (2) the validation-selected threshold, (3) test mAP@0.5 and mAP@0.5:0.95, (4) image-level precision, recall, F1, and target-absent false-positive rate, and (5) the limitation that all evaluated sequences come from Dataset 1's labelled training partition.

Do not compare this result directly with the Dataset 2 bird-hard-negative result as though the negative cohorts were interchangeable. The next combined experiment should preserve source identity in metadata and report source-specific test results in addition to an overall combined result.